In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score

In [3]:
df = pd.read_csv('Indian_Insurance_Data.csv')

In [4]:
df.shape

(4000, 8)

In [5]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
1558,58,99,172,48.14,False,Vadodara,Software Engineer,High
335,49,49,162,28.52,True,Ranchi,Shop Owner,High
3195,36,48,169,48.37,False,Rajkot,Nurse,Medium
2924,58,58,171,38.78,False,Indore,Government Employee,Medium
3163,35,57,188,18.79,False,Varanasi,Civil Servant,Low


In [6]:
df['city'].unique()

array(['Agra', 'Allahabad', 'Srinagar', 'Meerut', 'Varanasi', 'Hyderabad',
       'Ahmedabad', 'Chennai', 'Amritsar', 'Vadodara', 'Lucknow',
       'Bhopal', 'Mumbai', 'Ludhiana', 'Rajkot', 'Surat', 'Ghaziabad',
       'Ranchi', 'Pune', 'Kanpur', 'Nagpur', 'Faridabad', 'Bangalore',
       'Jaipur', 'Kolkata', 'Nashik', 'Patna', 'Indore', 'Delhi',
       'Visakhapatnam'], dtype=object)

In [7]:
df['occupation'].unique()

array(['Factory Worker', 'Businessman', 'Sales Manager', 'Banker',
       'Marketing Manager', 'Insurance Agent', 'HR Manager', 'Pharmacist',
       'Teacher', 'Software Engineer', 'Consultant', 'Driver',
       'Shop Owner', 'Nurse', 'Accountant', 'Government Employee',
       'Architect', 'Engineer', 'Real Estate Agent', 'Civil Servant',
       'Plumber', 'Retail Manager', 'Chef', 'Electrician', 'Carpenter',
       'Doctor', 'Lab Technician', 'Data Analyst', 'Lawyer',
       'Content Writer'], dtype=object)

In [8]:
df_feat = df.copy()

In [9]:
df_feat['bmi'] = df_feat['weight'] / ((df_feat['height']/100) ** 2)
df_feat.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi
0,50,57,176,29.90,False,Agra,Factory Worker,Medium,18.401343
1,56,90,168,19.22,False,Allahabad,Businessman,Medium,31.887755
2,26,87,178,36.95,False,Srinagar,Sales Manager,Low,27.458654
3,34,73,162,14.75,False,Meerut,Banker,Low,27.815882
4,43,47,192,19.32,True,Varanasi,Marketing Manager,High,12.749566


In [10]:
def age_group(age):
  if age < 25:
    return 'Young Adult'
  elif age < 45:
    return 'Adult'
  elif age < 65:
    return 'Middle Age'
  else:
    return 'Senior'

In [11]:
df_feat["age_group"] = df_feat['age'].apply(age_group)

In [12]:
def lifestyle_risk(row):
  if row['smoker'] and row['bmi'] > 30:
    return 'High'
  elif row['smoker'] or row['bmi'] > 25:
    return 'Medium'
  else:
    return 'Low'


df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk, axis = 1)

In [13]:

tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [14]:
def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3


df_feat['city_tier'] = df_feat['city'].apply(city_tier)

In [15]:
df_feat.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
0,50,57,176,29.90,False,Agra,Factory Worker,Medium,18.401343,Middle Age,Low,2
1,56,90,168,19.22,False,Allahabad,Businessman,Medium,31.887755,Middle Age,Medium,2
2,26,87,178,36.95,False,Srinagar,Sales Manager,Low,27.458654,Adult,Medium,3
3,34,73,162,14.75,False,Meerut,Banker,Low,27.815882,Adult,Medium,3
4,43,47,192,19.32,True,Varanasi,Marketing Manager,High,12.749566,Adult,Medium,2


In [16]:
df_feat.drop(columns = ['age', 'weight', 'height', 'smoker', 'city'])

,income_lpa,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
0,29.90,Factory Worker,Medium,18.401343,Middle Age,Low,2
1,19.22,Businessman,Medium,31.887755,Middle Age,Medium,2
2,36.95,Sales Manager,Low,27.458654,Adult,Medium,3
3,14.75,Banker,Low,27.815882,Adult,Medium,3
4,19.32,Marketing Manager,High,12.749566,Adult,Medium,2
...,...,...,...,...,...,...,...
3995,37.65,Insurance Agent,Medium,23.624447,Adult,Medium,2
3996,10.00,Data Analyst,Low,28.360352,Adult,Medium,2
3997,28.63,Chef,High,17.258941,Adult,Medium,2
3998,12.01,Pharmacist,Low,17.270698,Adult,Low,3


In [17]:
df_feat = df_feat[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']]

In [18]:
df_feat.sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
2065,47.17,Architect,28.934069,Adult,Medium,2,High
2009,32.12,Content Writer,22.545959,Adult,Low,2,Low
3114,10.70,Content Writer,24.772097,Middle Age,Medium,1,High
564,25.65,Insurance Agent,27.465303,Middle Age,Medium,2,Medium
2780,25.34,Factory Worker,26.304602,Middle Age,Medium,3,Medium


In [19]:
x = df_feat[['bmi', 'occupation', 'age_group', 'lifestyle_risk', 'city_tier', 'income_lpa']]
y = df_feat['insurance_premium_category']

In [20]:
y

,insurance_premium_category
0,Medium
1,Medium
2,Low
3,Low
4,High
...,...
3995,Medium
3996,Low
3997,High
3998,Low


In [21]:
df_feat.isnull().sum()

,0
income_lpa,0
occupation,0
bmi,0
age_group,0
lifestyle_risk,0
city_tier,0
insurance_premium_category,0


In [22]:
df_feat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   income_lpa                  4000 non-null   float64
 1   occupation                  4000 non-null   object 
 2   bmi                         4000 non-null   float64
 3   age_group                   4000 non-null   object 
 4   lifestyle_risk              4000 non-null   object 
 5   city_tier                   4000 non-null   int64  
 6   insurance_premium_category  4000 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 218.9+ KB


In [23]:
categorical_features = ['age_group', 'occupation', 'lifestyle_risk', 'city_tier']
numeric_features = ['bmi', 'income_lpa']

In [24]:
preprocessor = ColumnTransformer(
    transformers = [
        ('cat' , OneHotEncoder(), categorical_features),
        ('num', "passthrough", numeric_features)
    ]
)

In [25]:
pipeline = Pipeline(steps = [
    ('preprocessor' , preprocessor),
    ('classifier' , RandomForestClassifier( random_state = 42))
])

In [26]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1)
pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['bmi','occupation','age_group','lifestyle_risk','city_tier','income_lpa']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remain

In [27]:
#predict and evalutation
y_pred = pipeline.predict(x_test)
accuracy = accuracy_score(y_test,y_pred)

In [28]:
accuracy

0.74125

In [29]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        High       0.73      0.66      0.70       190
         Low       0.84      0.77      0.80       282
      Medium       0.68      0.76      0.72       328

    accuracy                           0.74       800
   macro avg       0.75      0.73      0.74       800
weighted avg       0.75      0.74      0.74       800



In [30]:
x_test.sample(5)

,bmi,occupation,age_group,lifestyle_risk,city_tier,income_lpa
2403,16.066482,Data Analyst,Adult,Medium,3,45.42
1742,25.510204,Driver,Adult,Medium,1,12.85
804,21.066743,Chef,Adult,Medium,1,8.19
2978,19.100092,Doctor,Young Adult,Low,2,46.34
1766,32.830980,Shop Owner,Middle Age,Medium,2,12.32


In [31]:
y.value_counts()

,count
insurance_premium_category,
Medium,1656
Low,1337
High,1007


In [32]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred)

array([[126,   4,  60],
       [  5, 217,  60],
       [ 41,  37, 250]])

A Random Forest classifier with preprocessing using a scikit-learn Pipeline and ColumnTransformer achieved an accuracy of 74.1%. Given the limited feature set, this serves as a strong baseline. Further improvements would likely require additional predictive features, better feature engineering or hyperparameter tuning rather than changes to the preprocessing pipeline

In [33]:
import pickle

pickle_model_path = 'insurance_model.pkl'
with open(pickle_model_path, 'wb') as f:
  pickle.dump(pipeline, f)

In [34]:
df_feat['occupation'].unique()

array(['Factory Worker', 'Businessman', 'Sales Manager', 'Banker',
       'Marketing Manager', 'Insurance Agent', 'HR Manager', 'Pharmacist',
       'Teacher', 'Software Engineer', 'Consultant', 'Driver',
       'Shop Owner', 'Nurse', 'Accountant', 'Government Employee',
       'Architect', 'Engineer', 'Real Estate Agent', 'Civil Servant',
       'Plumber', 'Retail Manager', 'Chef', 'Electrician', 'Carpenter',
       'Doctor', 'Lab Technician', 'Data Analyst', 'Lawyer',
       'Content Writer'], dtype=object)

In [35]:
pd.crosstab(df_feat["occupation"], df_feat["insurance_premium_category"])

insurance_premium_category,High,Low,Medium
occupation,,,
Accountant,33,59,66
Architect,25,36,51
Banker,42,46,50
Businessman,32,30,41
Carpenter,35,39,61
Chef,31,44,51
Civil Servant,33,45,70
Consultant,32,53,48
Content Writer,38,43,55


In [36]:
import sklearn
print(sklearn.__version__)

1.9.0
